# Демо 3. RAG на LangChain і оцінювання

У демо 2 ми написали RAG-пайплайн вручну. Тут збираємо той самий пайплайн із готових компонентів
[LangChain](https://python.langchain.com/), додаємо два способи покращити пошук — **гібридний пошук**
і **переранжування** — і, головне, **вимірюємо** якість системи на наборі питань із відомими відповідями.

Ретрив і генерацію оцінюємо окремо:
- **ретрив** — чи потрапляє фрагмент із потрібного документа до знайдених (hit rate@k, MRR);
- **генерація** — чи правильна відповідь, чи посилається вона на правильне джерело
  і чи відмовляється система відповідати, коли відповіді в документах немає.

_Для цього ноутбука потрібен GPU ._

In [1]:
# LangChain розбитий на пакети: core — базові абстракції, text-splitters — розбиття тексту,
# community — інтеграції (FAISS, BM25), huggingface — моделі з Hugging Face
!pip install -q "sentence-transformers>=5" pypdf langchain-core langchain-text-splitters langchain-community langchain-huggingface faiss-cpu rank_bm25

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

/tmp/ipykernel_1111/1402898701.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Пристрій: cuda


## 1. Документи

Ті самі 14 документів, що в демо 1 і 2. LangChain має готові завантажувачі PDF, але колонтитули
й скани все одно доводиться обробляти самостійно, тому текст витягуємо так само, як раніше,
а потім загортаємо кожен документ в обʼєкт `Document`: текст плюс метадані.

In [3]:
# @title Завантаження й очищення документів (як у демо 1)
BASE_URL = "https://docs.ztu.edu.ua/"
DOCUMENTS = [
    ("275", "Положення про оцінювання результатів навчання (чинна версія 4.0)", {"mdocs-file": 275}),
    ("275-v3.1", "Положення про оцінювання результатів навчання (стара версія 3.1)",
     {"mdocs-version": "polozhennja pro ocinjuvannja znan yekts_nove.pdf-v3.1", "mdocs-file": 275, "mdocs-url": "false"}),
    ("277", "Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін", {"mdocs-file": 277}),
    ("291", "Положення про проведення практики здобувачів вищої освіти", {"mdocs-file": 291}),
    ("340", "Правила призначення академічних стипендій", {"mdocs-file": 340}),
    ("344", "Положення про стипендію ректора", {"mdocs-file": 344}),
    ("348", "Положення про користування гуртожитками", {"mdocs-file": 348}),
    ("1219", "Правила внутрішнього розпорядку в студентських гуртожитках", {"mdocs-file": 1219}),
    ("1199", "Кодекс академічної доброчесності", {"mdocs-file": 1199}),
    ("4389", "Положення про студентського омбудсмана", {"mdocs-file": 4389}),
    ("336", "Положення про старосту академічної групи", {"mdocs-file": 336}),
    ("3320", "ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року", {"mdocs-file": 3320}),
    ("3329", "ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року", {"mdocs-file": 3329}),
    ("283", "Положення про переведення здобувачів вищої освіти", {"mdocs-file": 283}),
]


PDF_DIR = Path("ztu_pdfs")
PDF_DIR.mkdir(exist_ok=True)


def fetch(params, attempts=3):
    # Сайт іноді обриває зʼєднання, тому пробуємо кілька разів
    for attempt in range(attempts):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(5)


def download_pdf(doc_id, params):
    # Зберігаємо PDF на диск, щоб під час повторного запуску не завантажувати їх знову
    path = PDF_DIR / f"{doc_id}.pdf"
    if not path.exists():
        path.write_bytes(fetch(params))
    return PdfReader(path)


HEADER_LINE = re.compile(
    r"^(Житомирська|політехніка|Міністерство освіти.*|Державний університет «Житомирська політехніка»|"
    r"Система управління якістю.*|П[\s\-–]*[CС]?[АA]?[ГG]?[\s\d.\-–]*|[\d.\-–\s/_]+|Випуск.*)$",
    re.IGNORECASE,
)
HEADER_LINES = 12


def page_text(page):
    lines = (page.extract_text() or "").split("\n")
    head = [line for line in lines[:HEADER_LINES] if not HEADER_LINE.match(line.strip())]
    return " ".join(head + lines[HEADER_LINES:])


MIN_CHARS_PER_PAGE = 100


def document_text(reader):
    return re.sub(r"\s+", " ", " ".join(page_text(page) for page in reader.pages)).strip()


def is_scan(text, reader):
    return len(text) < MIN_CHARS_PER_PAGE * len(reader.pages)


documents = {}
for doc_id, title, params in DOCUMENTS:
    reader = download_pdf(doc_id, params)
    text = document_text(reader)
    if is_scan(text, reader):
        print(f"{doc_id}: текстового шару немає, потрібне OCR — пропущено")
        continue
    documents[doc_id] = {"title": title, "text": text}
    print(f"{doc_id}: {len(reader.pages)} сторінок, {len(text)} символів — {title}")

275: 23 сторінок, 24944 символів — Положення про оцінювання результатів навчання (чинна версія 4.0)
275-v3.1: 13 сторінок, 10505 символів — Положення про оцінювання результатів навчання (стара версія 3.1)
277: 16 сторінок, 15945 символів — Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін
291: 17 сторінок, 14603 символів — Положення про проведення практики здобувачів вищої освіти
340: 30 сторінок, 35706 символів — Правила призначення академічних стипендій
344: 10 сторінок, 2455 символів — Положення про стипендію ректора
348: 22 сторінок, 25608 символів — Положення про користування гуртожитками
1219: 12 сторінок, 9159 символів — Правила внутрішнього розпорядку в студентських гуртожитках
1199: 37 сторінок, 56918 символів — Кодекс академічної доброчесності
4389: 18 сторінок, 20685 символів — Положення про студентського омбудсмана
336: 8 сторінок, 3560 символів — Положення про старосту академічної групи
3320: 16 сторінок, 31670 символів — ОПП F2 «Інженері

In [4]:
docs = [
    Document(page_content=doc["text"], metadata={"doc_id": doc_id, "title": doc["title"]})
    for doc_id, doc in documents.items()
]
print(docs[0].metadata, docs[0].page_content[:200])

{'doc_id': '275', 'title': 'Положення про оцінювання результатів навчання (чинна версія 4.0)'} ЗАТВЕРДЖЕНО Наказ Державного університету «Житомирська політехніка» 15 вересня 2025 р. № 327/од Ректор Віктор ЄВДОКИМОВ ПОЛОЖЕННЯ про оцінювання результатів навчання здобувачів вищої освіти у Державно


## 2. Розбиття на фрагменти

`RecursiveCharacterTextSplitter` пробує різати текст за першим роздільником зі списку; якщо шматок
усе ще задовгий, переходить до наступного роздільника. Наш список: кінець речення, крапка з комою,
пробіл і, в крайньому разі, будь-який символ. Метадані документа копіюються в кожен фрагмент.
Як і раніше, дописуємо на початок фрагмента назву документа.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, separators=[". ", "; ", " ", ""], keep_separator="end"
)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    chunk.page_content = f"{chunk.metadata['title']}. {chunk.page_content}"

print(f"Фрагментів: {len(chunks)}")
print(chunks[100].metadata)
print(chunks[100].page_content)

Фрагментів: 385
{'doc_id': '340', 'title': 'Правила призначення академічних стипендій'}
Правила призначення академічних стипендій. 1.9. Студенти, аспіранти (докторанти), які реалізують право на академічну мобільність і зберігають статус здобувача вищої освіти (наукового ступеня) за денною формою навчання (навчаються з відривом від виробництва) у Житомирській політехніці, протягом строку навчання за програмою академічної мобільності в іншому навчальному закладі поза межами України отримують стипендію, призначену відповідно до цих Правил, у разі, коли умовами договору про навчання за програмою академічної мобільності, укладеного ними з Житомирською політехнікою, регулярна безповоротна фіксована фінансова підтримка у грошовій формі протягом всього строку навчання за програмою академічної мобільності в іншому навчальному закладі не передбачена або передбачена у розмірі, що у перерахунку на національну грошову одиницю на дату укладення договору про академічну мобільність є меншою за розмір 

## 3. Векторне сховище

`HuggingFaceEmbeddings` — обгортка LangChain над sentence-transformers. Для документів і запитів
задаємо різні параметри: запит кодується з промптом `query`, як у демо 1.
[FAISS](https://github.com/facebookresearch/faiss) — бібліотека векторного пошуку від Meta;
для нормалізованих векторів обираємо скалярний добуток як міру подібності.

In [6]:
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device, "model_kwargs": {"dtype": dtype}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    query_encode_kwargs={"normalize_embeddings": True, "prompt_name": "query"},
)
vector_store = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

for doc, score in vector_store.similarity_search_with_score("Коли обирають старосту на першому курсі?", k=3):
    print(f"{score:.2f}  [{doc.metadata['doc_id']}] {doc.page_content[:120]}…")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

0.60  [336] Положення про старосту академічної групи. У своїй роботі староста академічної групи співпрацює з куратором та ментором г…
0.53  [336] Положення про старосту академічної групи. До виборів обов’язки старости групи може виконувати один із студентів групи за…
0.50  [4389] Положення про студентського омбудсмана. 6.4. Студентський омбудсман взаємодіє з Молодіжною радою при Житомирській обласн…


## 4. Лексичний пошук BM25

BM25 — класичний алгоритм пошуку за збігом слів: фрагмент отримує тим більшу оцінку, чим більше в ньому
слів запиту і чим рідше ці слова трапляються в інших фрагментах. BM25 не розуміє синонімів і,
без додаткової обробки, різних форм того самого слова («гуртожиток» і «гуртожитку» для нього різні слова),
зате точно знаходить рідкісні слова, номери й коди.

In [7]:
CANDIDATES = 20


def tokenize(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Retriever.from_documents(chunks, k=CANDIDATES, preprocess_func=tokenize)


def vector_search(query, k):
    return vector_store.similarity_search(query, k=k)


def bm25_search(query, k):
    return bm25.invoke(query)[:k]


def show(search, query, k=3):
    print(f"{search.__name__}: {query}")
    for doc in search(query, k):
        print(f"  [{doc.metadata['doc_id']}] {doc.page_content[:110]}…")
    print()


for query in ["До якої дати потрібно оплатити гуртожиток за перший семестр?",
              "Яку допомогу отримує студент в академічній відпустці за медичними показаннями?"]:
    show(vector_search, query)
    show(bm25_search, query)

vector_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 5.2. Студент вносить авансовий платіж оплати за 1 семестр (5 місяців)…
  [348] Положення про користування гуртожитками. Мешканці гуртожитків Житомирської політехніки зобов’язані: - знати й …
  [348] Положення про користування гуртожитками. Це є єдиною підставою для поселення і проживання студента у вказаному…

bm25_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 3.9. Здобувачам освіти, які поселяються до гуртожитку Житомирської по…
  [340] Правила призначення академічних стипендій. Цей показник встановлюється у відсотках (у діапазоні від 40 до 45) …
  [340] Правила призначення академічних стипендій. Рішенням ректора Житомирської політехніки таким особам може встанов…

vector_search: Яку допомогу отримує студент в академічній відпустці за медичними показаннями?
  [340] Правила призначення академ

Фрагмент з відповіддю на перше питання містить слова «авансовий платіж оплати… до 1 вересня»,
а не «оплатити»: семантичний пошук знаходить його, а BM25 — лише серед нижчих результатів.
У другому питанні рідкісний вираз «за медичними показаннями» дослівно є у фрагменті з відповіддю,
і BM25 ставить цей фрагмент першим, а семантичний пошук — нижче.

In [20]:
# Питання 24: власні питання, на яких BM25 і семантичний пошук дають різні результати
OWN_QUESTIONS = [
    ("Якій оцінці відповідають 90-100 балів?", ["275"], r"90 ?– ?100"),
    ("Скільки грошей треба внести за проживання перед заселенням?", ["348"], r"авансов\w* платіж"),
    ("Чи можна приводити друзів у кімнату вночі?", ["1219", "348"], r"0?8:00 до 22:00"),
]


def rank_of(search, question, gold, evidence):
    for rank, doc in enumerate(search(question, 20), start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content):
            return rank
    return None


for question, gold, evidence in OWN_QUESTIONS:
    print(question)
    for search in (vector_search, bm25_search):
        top = search(question, 1)[0]
        print(f"  {search.__name__}: місце правильного фрагмента {rank_of(search, question, gold, evidence)}, перший результат [{top.metadata['doc_id']}] {top.page_content[:110]}…")
    print()

Якій оцінці відповідають 90-100 балів?
  vector_search: місце правильного фрагмента 9, перший результат [275-v3.1] Положення про оцінювання результатів навчання (стара версія 3.1). 2.4. Кожний вид навчальної активності здобув…
  bm25_search: місце правильного фрагмента 4, перший результат [340] Правила призначення академічних стипендій. У разі наявності двох або однієї особи на курсі за певною спеціальн…

Скільки грошей треба внести за проживання перед заселенням?
  vector_search: місце правильного фрагмента 3, перший результат [348] Положення про користування гуртожитками. 4.15. Самовільне заселення у гуртожиток не допускається і тягне за со…
  bm25_search: місце правильного фрагмента 14, перший результат [348] Положення про користування гуртожитками. 4.7. За порушення Правил внутрішнього розпорядку гуртожитку адміністр…

Чи можна приводити друзів у кімнату вночі?
  vector_search: місце правильного фрагмента 1, перший результат [1219] Правила внутрішнього розпорядку в студентських гур

## 5. Гібридний пошук

Два списки результатів обʼєднуємо методом **reciprocal rank fusion** (RRF): фрагмент на позиції $r$
у кожному списку отримує $\frac{1}{60 + r}$ балів, бали з обох списків додаються.
RRF використовує лише позиції, тому не потребує зведення оцінок BM25 і косинусної подібності до однієї шкали.

In [8]:
def rrf(rankings, c=60):
    scores, by_content = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (c + rank)
            by_content[doc.page_content] = doc
    return [by_content[content] for content in sorted(scores, key=scores.get, reverse=True)]


def hybrid_search(query, k):
    return rrf([vector_search(query, CANDIDATES), bm25_search(query, CANDIDATES)])[:k]


show(hybrid_search, "Якому діапазону балів відповідає оцінка FX?")

hybrid_search: Якому діапазону балів відповідає оцінка FX?
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Під час поточного контролю оцінюванню можуть…



In [21]:
# Питання 25: RRF вручну для фрагмента, який у векторному пошуку стоїть на 2 місці, а в BM25 — на 5
c = 60
vector_part = 1 / (c + 2)
bm25_part = 1 / (c + 5)
print(f"внесок векторного пошуку: 1/(60+2) = {vector_part:.5f}")
print(f"внесок BM25:             1/(60+5) = {bm25_part:.5f}")
print(f"разом:                              {vector_part + bm25_part:.5f}")
print(f"для порівняння, 1 місце лише в одному списку: 1/(60+1) = {1 / (c + 1):.5f}")

# Перевірка функцією rrf на невеликих списках: потрібний фрагмент T стоїть на 2 місці у першому списку і на 5 в другому
names = ["A", "T", "C", "D", "E"], ["F", "G", "H", "I", "T"]
lists = [[Document(page_content=n) for n in group] for group in names]
print([doc.page_content for doc in rrf(lists)][:3])

# Чому важливі саме позиції: шкали оцінок двох методів різні
query = "Якому діапазону балів відповідає оцінка FX?"
print("найкраща оцінка векторного пошуку:", round(float(vector_store.similarity_search_with_score(query, k=1)[0][1]), 2))
print("найкраща оцінка BM25:", round(float(max(bm25.vectorizer.get_scores(tokenize(query)))), 2))

внесок векторного пошуку: 1/(60+2) = 0.01613
внесок BM25:             1/(60+5) = 0.01538
разом:                              0.03151
для порівняння, 1 місце лише в одному списку: 1/(60+1) = 0.01639
['T', 'A', 'F']
найкраща оцінка векторного пошуку: 0.39
найкраща оцінка BM25: 8.5


## 6. Переранжування

Модель ембедингів — **bi-encoder**: запит і фрагмент кодуються окремо, тому ембединги фрагментів можна порахувати
заздалегідь. **Cross-encoder** отримує запит і фрагмент разом і оцінює їхню відповідність точніше,
але для кожної пари потрібен окремий прохід моделі. Тому cross-encoder застосовують лише до кількох десятків
кандидатів, знайдених швидким пошуком.
Використовуємо багатомовну модель [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3).

In [9]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})


def rerank_search(query, k):
    candidates = hybrid_search(query, CANDIDATES)
    scores = reranker.predict([(query, doc.page_content) for doc in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]


show(rerank_search, "Якому діапазону балів відповідає оцінка FX?")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

rerank_search: Якому діапазону балів відповідає оцінка FX?
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Значення вагового коефіцієнта для кожного ви…



In [22]:
# Питання 26: скільки часу займає cross-encoder на 20 кандидатах і на всіх фрагментах корпусу
query = "Якому діапазону балів відповідає оцінка FX?"
candidates = hybrid_search(query, CANDIDATES)

start = time.time()
reranker.predict([(query, doc.page_content) for doc in candidates])
t_20 = time.time() - start

start = time.time()
reranker.predict([(query, doc.page_content) for doc in chunks])
t_all = time.time() - start

print(f"кандидатів: {len(candidates)}, час: {t_20:.2f} с")
print(f"усіх фрагментів: {len(chunks)}, час: {t_all:.2f} с")
print(f"на одну пару: {t_all / len(chunks) * 1000:.0f} мс; для 26 питань бенчмарку без відбору: {t_all * 26:.0f} с проти {t_20 * 26:.0f} с")

кандидатів: 20, час: 0.34 с
усіх фрагментів: 385, час: 4.59 с
на одну пару: 12 мс; для 26 питань бенчмарку без відбору: 119 с проти 9 с


## 7. Бенчмарк

Набір питань складено **до** налаштування системи. Для кожного питання вказано документи, у яких є відповідь,
регулярний вираз, за яким можна впізнати фрагмент із відповіддю, та еталонну відповідь. Три питання не мають відповіді в документах: на них система має відмовитися відповідати.

In [10]:
BENCHMARK = [
    ("До якої дати потрібно оплатити гуртожиток за перший семестр?", ["348"], r"1 вересня", "До 1 вересня поточного року."),
    ("До якої дати оплачують гуртожиток за другий семестр?", ["348"], r"1 лютого", "До 1 лютого поточного року."),
    ("З якої до якої години в гуртожитку має бути тиша?", ["1219"], r"23:00 години і до 7:00", "З 23:00 до 7:00."),
    ("У який час гості можуть перебувати в гуртожитку?", ["1219", "348"], r"0?8:00 до 22:00", "З 8:00 до 22:00."),
    ("Чи можна тримати в гуртожитку домашню тварину?", ["1219", "348"], r"утримувати тварин", "Ні, утримувати тварин у гуртожитку заборонено."),
    ("На який строк укладають договір найму житла в гуртожитку?", ["348"], r"терміном на один навчальний рік", "На один навчальний рік."),
    ("За який час потрібно звільнити гуртожиток після відрахування?", ["1219"], r"протягом двох тижнів", "Протягом двох тижнів від дня виходу наказу."),
    ("Коли обирають старосту на першому курсі?", ["336"], r"після двох тижнів", "На початку навчального року, після двох тижнів навчального процесу."),
    ("На який строк обирається староста групи?", ["336"], r"на весь період навчання", "На весь період навчання."),
    ("Скільки балів потрібно для оцінки «добре» B?", ["275"], r"82 ?– ?89", "82–89 балів."),
    ("Якому діапазону балів відповідає оцінка FX?", ["275"], r"35 ?– ?59", "35–59 балів."),
    ("Яку частку кредитів становлять вибіркові дисципліни?", ["277"], r"25 відсотків",
     "Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням)."),
    ("До якої дати викладачі вносять анотації вибіркових дисциплін?", ["277"], r"30 квітня", "До 30 квітня."),
    ("Як часто виплачують стипендію?", ["340"], r"один раз на місяць", "Один раз на місяць."),
    ("За яким показником формують рейтинг для призначення академічної стипендії?", ["340"], r"середнім балом", "За середнім балом семестрового контролю."),
    ("Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної",
     "Щомісячну допомогу в розмірі 50 відсотків мінімальної академічної стипендії."),
    ("Для кого встановлюється стипендія ректора?", ["344"], r"встановлюється для студентів денної",
     "Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності."),
    ("На який строк обирають студентського омбудсмана?", ["4389"], r"терміном на 1 рік", "На 1 рік."),
    ("До якої дати проводять вибори студентського омбудсмана?", ["4389"], r"до 01 жовтня", "До 1 жовтня кожного навчального року."),
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат –", "Оприлюднення власних раніше опублікованих результатів без зазначення джерела попереднього оприлюднення."),
    ("Що таке фабрикація в академічній діяльності?", ["1199"], r"Фабрикація –", "Вигадування даних чи фактів про результати власної академічної діяльності."),
    ("Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС?", ["3320", "3329"], r"240 кредит", "240 кредитів ЄКТС."),
    ("Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?", ["3320"], r"спеціальн\w* F2", "F2."),
    ("Скільки коштує обід у студентській їдальні?", [], None, None),
    ("Який графік роботи бібліотеки університету?", [], None, None),
    ("Скільки коштує рік навчання на контракті за спеціальністю «Інженерія програмного забезпечення»?", [], None, None),
]
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")

Питань: 26, з них без відповіді в документах: 3


### Метрики ретриву

Фрагмент вважаємо правильним, якщо він походить із правильного документа і містить вираз-доказ.
Перевіряти лише документ замало: у довгому документі десятки фрагментів, і більшість із них відповіді не містять.

- **hit rate@k** — частка питань, для яких правильний фрагмент є серед `k` знайдених;
- **MRR** (mean reciprocal rank) — середнє за питаннями значення $\frac{1}{r}$, де $r$ — позиція першого
  правильного фрагмента; якщо його немає серед перших 10, замість $\frac{1}{r}$ береться 0. MRR враховує не лише те, чи знайдено фрагмент, а й те, наскільки високо.

Питання без відповіді в ретриві не оцінюємо: правильного фрагмента для них немає.

In [11]:
def first_hit_rank(results, gold, evidence):
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content):
            return rank
    return None


def hit_rate(ranks, k):
    return np.mean([rank is not None and rank <= k for rank in ranks])


def mrr(ranks):
    return np.mean([1 / rank if rank else 0 for rank in ranks])


def retrieval_metrics(search):
    ranks = [first_hit_rank(search(question, 10), gold, evidence) for question, gold, evidence in answerable]
    return {"метод": search.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}


pd.DataFrame([retrieval_metrics(search) for search in [vector_search, bm25_search, hybrid_search, rerank_search]]).round(2)

,метод,hit@1,hit@5,MRR
0,vector_search,0.61,0.87,0.74
1,bm25_search,0.52,0.65,0.59
2,hybrid_search,0.61,0.91,0.76
3,rerank_search,0.83,0.96,0.90


In [23]:
# Питання 27: додаю до бенчмарку три власні питання (з відповіддю, перефразоване, без відповіді) і перераховую метрики
methods = [vector_search, bm25_search, hybrid_search, rerank_search]
before = pd.DataFrame([retrieval_metrics(search) for search in methods])

NEW_QUESTIONS = [
    ("Скільки місяців тривають розрахунки за перший семестр у гуртожитку?", ["348"], r"5 місяців", "5 місяців."),
    ("Чи дозволено заселяти в кімнату кота чи собаку?", ["1219", "348"], r"утримувати тварин", "Ні, утримувати тварин у гуртожитку заборонено."),
    ("Скільки коштує ксерокопія документів у деканаті?", [], None, None),
]
BENCHMARK += NEW_QUESTIONS
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
after = pd.DataFrame([retrieval_metrics(search) for search in methods])

print(f"Питань: {len(BENCHMARK)}, з відповіддю: {len(answerable)}")
print("До додавання:")
print(before.round(2).to_string(index=False))
print("Після додавання:")
print(after.round(2).to_string(index=False))
print()
for question, gold, evidence, _ in NEW_QUESTIONS[:2]:
    ranks = {search.__name__: first_hit_rank(search(question, 10), gold, evidence) for search in methods}
    print(question, ranks)

Питань: 29, з відповіддю: 25
До додавання:
        метод  hit@1  hit@5  MRR
vector_search   0.61   0.87 0.74
  bm25_search   0.52   0.65 0.59
hybrid_search   0.61   0.91 0.76
rerank_search   0.83   0.96 0.90
Після додавання:
        метод  hit@1  hit@5  MRR
vector_search   0.64   0.88 0.76
  bm25_search   0.52   0.64 0.58
hybrid_search   0.64   0.92 0.78
rerank_search   0.84   0.96 0.90

Скільки місяців тривають розрахунки за перший семестр у гуртожитку? {'vector_search': 1, 'bm25_search': 1, 'hybrid_search': 1, 'rerank_search': 1}
Чи дозволено заселяти в кімнату кота чи собаку? {'vector_search': 1, 'bm25_search': None, 'hybrid_search': 1, 'rerank_search': 1}


На українських текстах BM25 сам по собі зазвичай поступається семантичному пошуку: слова мають багато форм,
а питання рідко повторюють формулювання документа. Переранжування найсильніше впливає на hit@1:
cross-encoder бачить запит і фрагмент разом і частіше ставить правильний фрагмент на перше місце.
Висновки варто робити обережно: на 23 питаннях одне питання — це понад 4 відсоткові пункти.

## 8. RAG-ланцюжок на LangChain

Генеративна модель та сама, що в демо 2. Загортаємо її в `HuggingFacePipeline`, а `ChatHuggingFace`
застосовує до повідомлень шаблон чату моделі.

In [12]:
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)

generator = pipeline(
    "text-generation", model=llm, tokenizer=tokenizer,
    max_new_tokens=300, do_sample=False, temperature=None, top_p=None, top_k=None, return_full_text=False,
)
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator), tokenizer=tokenizer)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Ланцюжок записуємо мовою LCEL (LangChain Expression Language): оператор `|` передає вихід одного кроку
на вхід наступного. Ланцюжок повертає і відповідь, і знайдені фрагменти: вони потрібні для перевірки цитувань.

In [13]:
SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання студентів Житомирської політехніки. "
    "Відповідай лише на основі наданих фрагментів документів. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Якщо у фрагментах немає відповіді на питання, відповідай лише: «У наданих документах немає відповіді.» "
    "Відповідай українською мовою, коротко."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Фрагменти документів:\n\n{context}\n\nПитання: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{n}] {doc.page_content}" for n, doc in enumerate(docs, start=1))


retriever = RunnableLambda(lambda question: rerank_search(question, 5))
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | chat
    | StrOutputParser()
)
rag_chain = {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(answer=answer_chain)

result = rag_chain.invoke("Чи можна тримати в гуртожитку домашню тварину?")
print(result["answer"])
for n, doc in enumerate(result["docs"], start=1):
    print(f"  [{n}] {doc.metadata['doc_id']}: {doc.metadata['title']}")

Ні, в гуртожитку домашню тварину тримати заборонено [1].
  [1] 348: Положення про користування гуртожитками
  [2] 348: Положення про користування гуртожитками
  [3] 348: Положення про користування гуртожитками
  [4] 1219: Правила внутрішнього розпорядку в студентських гуртожитках
  [5] 1219: Правила внутрішнього розпорядку в студентських гуртожитках


## 9. Оцінювання відповідей

Для кожного питання бенчмарку перевіряємо:
- **правильність**: для питань із відповіддю — чи збігається відповідь з еталонною; для питань без відповіді — чи відмовилася система;
- **цитування**: чи є серед фрагментів, на які посилається відповідь, фрагмент із правильного документа.

Правильність вільної відповіді важко перевірити правилом: «до 1 вересня» і «не пізніше першого вересня» — одна відповідь.
Тому використовуємо підхід **LLM як суддя**: інша (тут — та сама) модель порівнює відповідь з еталоном.
Суддя теж помиляється, тому його вердикти варто вибірково перевіряти вручну.

In [24]:
REFUSAL = "немає відповіді"
JUDGE_PROMPT = (
    "Порівняй відповідь системи з еталонною відповіддю на питання.\n\n"
    "Питання: {question}\nЕталонна відповідь: {reference}\nВідповідь системи: {answer}\n\n"
    "Чи містить відповідь системи ту саму інформацію, що й еталонна, без суперечностей? "
    "Відповідай одним словом: «так» або «ні»."
)


def judge(question, reference, answer):
    verdict = chat.invoke(JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)).content
    return verdict.strip().lower().startswith("так")


def cited_doc_ids(answer, retrieved):
    # Номери на зразок [2] у відповіді перетворюємо на ідентифікатори документів
    numbers = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [retrieved[n - 1] for n in numbers if 0 < n <= len(retrieved)]


def evaluate(question, gold, reference):
    result = rag_chain.invoke(question)
    answer = result["answer"]
    retrieved = [doc.metadata["doc_id"] for doc in result["docs"]]
    if gold:
        correct = judge(question, reference, answer)
        citation_ok = any(doc_id in gold for doc_id in cited_doc_ids(answer, retrieved))
    else:
        correct = REFUSAL in answer.lower()
        citation_ok = None
    return {"питання": question, "правильно": correct, "цитування": citation_ok,
            "документи": gold, "знайдено": retrieved, "відповідь": answer}


evaluation = pd.DataFrame([evaluate(question, gold, reference) for question, gold, _, reference in BENCHMARK])
has_answer = evaluation["цитування"].notna()
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%}")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%}")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%}")

Правильних відповідей на питання з відповіддю: 92%
Коректних цитувань: 96%
Правильних відмов на питання без відповіді: 100%


In [27]:
# Питання 29: п'ять вердиктів судді для ручної перевірки
CHECK = [
    "Для кого встановлюється стипендія ректора?",
    "Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?",
    "Яку частку кредитів становлять вибіркові дисципліни?",
    "Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?",
    "Скільки балів потрібно для оцінки «добре» B?",
]
reference_by_question = {q: ref for q, _, _, ref in BENCHMARK}
for question in CHECK:
    row = evaluation[evaluation["питання"] == question].iloc[0]
    print(f"Питання: {question}")
    print(f"  еталон: {reference_by_question[question]}")
    print(f"  відповідь: {row['відповідь']}")
    print(f"  суддя вважає правильною: {row['правильно']}\n")

# Перевірка за текстом документів: чи є в положенні про стипендію ректора фраза про переможців олімпіад
for chunk in chunks:
    if chunk.metadata["doc_id"] == "344" and "переможцями" in chunk.page_content:
        start = chunk.page_content.find("встановлюється для студентів денної")
        print(chunk.page_content[start:start + 330])
        break

# і що кажуть про шифр обидві редакції програми
for chunk in chunks:
    if chunk.metadata["doc_id"] in ("3320", "3329") and re.search(r"спеціальн\w* (F2|121)", chunk.page_content):
        match = re.search(r"спеціальн\w* (F2|121)", chunk.page_content)
        print(chunk.metadata["doc_id"], chunk.page_content[max(0, match.start() - 40):match.end() + 40].replace("\n", " "))

Питання: Для кого встановлюється стипендія ректора?
  еталон: Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності.
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].
  суддя вважає правильною: False

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  еталон: F2.
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].
  суддя вважає правильною: False

Питання: Яку частку кредитів становлять вибіркові дисципліни?
  еталон: Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням).
  відповідь: Вибіркові дисципліни становлять не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсо

## 10. Аналіз відмов

Найкорисніша частина оцінювання — не підсумкове число, а розбір помилок. Для кожної неправильної відповіді
дивимося, чи був серед знайдених фрагментів правильний документ:
- якщо **не був** — це промах ретриву, і виправляти треба пошук (розбиття, модель ембедингів, гібридний пошук);
- якщо **був** — модель не скористалася контекстом або скористалася неправильно, і виправляти треба промпт чи модель.

In [25]:
def retrieval_verdict(row):
    if not row["документи"] or any(doc_id in row["документи"] for doc_id in row["знайдено"]):
        return "ретрив знайшов"
    return "промах ретриву"


def describe_failure(row):
    print(f"Питання: {row['питання']}")
    print(f"  правильний документ: {row['документи']}, знайдено: {row['знайдено']} — {retrieval_verdict(row)}")
    print(f"  правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"  відповідь: {row['відповідь']}\n")


failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
for _, row in failures.iterrows():
    describe_failure(row)

Питання: Для кого встановлюється стипендія ректора?
  правильний документ: ['344'], знайдено: ['344', '340', '344', '340', '340'] — ретрив знайшов
  правильно: False, цитування: True
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  правильний документ: ['3320'], знайдено: ['3329', '3329', '3320', '3329', '3329'] — ретрив знайшов
  правильно: False, цитування: False
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].



Кожну неправильну відповідь відносять до однієї з двох груп:
- **помилка оцінювання**: відповідь правильна, але суддя помилився (відповідь сформульована інакше або містить більше деталей, ніж еталон)
  або еталон неоднозначний (відповідь спирається на інше місце документа, ніж еталон, і не суперечить документу);
- **помилка системи**: промах ретриву, галюцинація попри наявний контекст або суперечливі джерела —
  як у питанні про шифр спеціальності, де чинна редакція програми сама містить старий шифр 121 (див. демо 2).

Перед тим як змінювати систему, варто перевірити, чи правильно оцінено відповідь.

## Підсумок

- LangChain дає готові компоненти — розбивачі тексту, обгортки моделей, векторні сховища, ретривери — і спосіб
  зʼєднати їх у ланцюжок; логіку пошуку, як-от гібридний пошук чи переранжування, легко додати власними функціями.
- BM25 і семантичний пошук помиляються на різних питаннях, тому їхнє поєднання зазвичай краще за кожен окремо.
- Cross-encoder точніше оцінює відповідність фрагмента запиту, але працює лише на невеликій кількості кандидатів.
- Ретрив і генерацію оцінюють окремо: так видно, де саме виникає помилка.
- Для автоматичного оцінювання відповідей існують готові бібліотеки, наприклад [RAGAS](https://docs.ragas.io/);
  вони теж спираються на LLM як суддю.

# Відповіді на питання 21–30

**21.** Весь ноутбук на T4 виконався приблизно за 10 хвилин. Найдовше йшли завантаження моделей і оцінювання. Генеративна модель Qwen3-4B завантажувалась понад 2 хвилини, а клітинка з оцінюванням 29 питань працювала близько 3 хвилин, бо на кожне питання окремо виконуються пошук, переранжування, генерація відповіді і ще один виклик моделі-судді. Помилок, через які виконання зупинялось, не було, але з'явилось кілька попереджень. Після `pip install` Colab вивів повідомлення про конфлікт версій: `google-colab` вимагає `requests==2.32.4`, а встановилась версія 2.34.2. Далі все працювало, тому я цього не виправляв. `langchain-community` позначено як бібліотеку, яка більше не розвивається. Hugging Face попереджає, що запити йдуть без токена, і через це завантаження моделей повільніші. `transformers` радить передавати дані пакетом, а не викликати pipeline по одному запиту. Окрема незручність у тому, що PDF і ваги моделей зберігаються лише на диску сеансу, тому після скидання сеансу все довелося б завантажувати наново.

**22.** `RecursiveCharacterTextSplitter` ріже текст за списком роздільників по черзі: спочатку за кінцем речення (`. `), якщо шматок усе ще довший за 1000 символів — за крапкою з комою, потім за пробілом і, зрештою, за будь-яким символом. Шматки склеюються до 1000 символів із перекриттям 200, тому фрагменти мають приблизно однакову довжину (у нас вийшло 385 фрагментів). Розбиття на речення з демо 2 (410 фрагментів) складає фрагмент із цілих речень, тому межа завжди проходить по реченню, а довжина фрагмента залежить від довжини речень. Рекурсивний розбивач, якщо речення задовге, переходить до `;` або до пробілу, тому фрагмент не виходить завеликим, хоча речення іноді розривається посередині. Параметр `keep_separator="end"` залишає роздільник у кінці попереднього шматка. Без нього крапка або губиться, або переходить на початок наступного фрагмента, і фрагмент починається з «. ». У наших документах є нумерація на зразок «5.2.» і дати, тому крапка в кінці речення потрібна, щоб текст фрагмента лишався читабельним.

**23.** Ембединги в цьому ноутбуці нормалізовані (`normalize_embeddings=True`), тобто довжина кожного вектора дорівнює 1. Для таких векторів скалярний добуток збігається з косинусною подібністю, тому більше значення означає ближчий за змістом фрагмент. Саме це й вимагає `MAX_INNER_PRODUCT`: FAISS шукає фрагменти з найбільшим скалярним добутком, і для нормалізованих векторів це дає правильний порядок. Наприклад, для питання про оцінку FX найкраща оцінка векторного пошуку була 0.39, і вона не може перевищити 1. Для ненормалізованих ембедингів так зробити можна, але результат буде іншим. Скалярний добуток росте разом із довжиною вектора, тому фрагмент з довшим вектором отримував би вищу оцінку незалежно від змісту. Щоб отримати косинусну подібність, вектори потрібно спочатку нормалізувати. Евклідова відстань для нормалізованих векторів дає той самий порядок, що й скалярний добуток (квадрат відстані дорівнює 2 мінус подвоєний скалярний добуток), а для ненормалізованих порядок би розійшовся.

**24.** Приклад, де BM25 кращий: «Якій оцінці відповідають 90-100 балів?». Правильний фрагмент у BM25 на 4 місці, у семантичному пошуку — на 9. У питанні є числа 90 і 100, які в таблиці оцінок стоять точно в такому ж вигляді, тому BM25 їх знаходить за збігом слів. Ембединги погано розрізняють числа, і семантичний пошук першими поставив фрагменти про шкалу оцінювання в старій версії положення 3.1. Приклади, де кращий семантичний пошук: «Скільки грошей треба внести за проживання перед заселенням?» (3 місце проти 14 у BM25) і «Чи можна приводити друзів у кімнату вночі?» (1 місце, а BM25 не знайшов правильний фрагмент серед 20 результатів). У документі стоїть «авансовий платіж» і «гості можуть перебувати з 8:00 до 22:00», тобто там немає слів «заселення», «друзі» і «вночі». BM25 шукає лише збіг слів, а семантичний пошук знаходить фрагмент за змістом.

**25.** Для фрагмента на 2 місці у векторному пошуку і на 5 у BM25 (c = 60) отримуємо 1/(60+2) = 0.01613 і 1/(60+5) = 0.01538, разом 0.03151. Фрагмент, який стоїть на 1 місці лише в одному списку, отримує 1/61 = 0.01639, тобто приблизно вдвічі менше. Тому фрагмент, який є в обох списках, піднімається вище: у перевірці на невеликих списках функція `rrf` поставила його на перше місце. RRF використовує тільки позиції, бо оцінки двох методів мають різні шкали і їх не можна додавати. Для питання про FX найкраща оцінка векторного пошуку 0.39, а найкраща оцінка BM25 — 8.5, до того ж оцінка BM25 залежить від довжини запиту і від складу корпусу. Позиції в списках можна порівнювати без жодних перетворень.

**26.** Cross-encoder читає запит і фрагмент разом, тому його результат не можна порахувати заздалегідь, як ембединги фрагментів. Для кожної пари (питання, фрагмент) потрібен окремий прохід моделі. Для всього корпусу на одне питання це 385 проходів, за 20 кандидатів — 20. Я заміряв час на T4: 20 пар обробляються за 0.34 с, а всі 385 фрагментів — за 4.59 с (близько 12 мс на пару). На 29 питаннях бенчмарку це приблизно 10 секунд проти 133 секунд, а на великому корпусі різниця була б ще більшою, бо час росте разом із кількістю фрагментів. Є і зворотний бік: cross-encoder може лише переставити кандидатів, тому якщо швидкий пошук не поставив правильний фрагмент до перших 20, переранжування його вже не знайде.

**27.** Я додав до бенчмарку три питання. З відповіддю: «Скільки місяців тривають розрахунки за перший семестр у гуртожитку?» (правильний фрагмент усі чотири методи ставлять на 1 місце). Перефразоване: «Чи дозволено заселяти в кімнату кота чи собаку?» замість «Чи можна тримати в гуртожитку домашню тварину?» (векторний пошук, гібридний і переранжування ставлять правильний фрагмент на 1 місце, BM25 не знаходить його серед 10, бо в документі немає слів «кіт» і «собака»). Без відповіді: «Скільки коштує ксерокопія документів у деканаті?». Метрики ретриву рахуються лише для питань із відповіддю, тому їх стало 25 замість 23. Було → стало (hit@1 / hit@5 / MRR): векторний 0.61/0.87/0.74 → 0.64/0.88/0.76, BM25 0.52/0.65/0.59 → 0.52/0.64/0.58, гібридний 0.61/0.91/0.76 → 0.64/0.92/0.78, з переранжуванням 0.83/0.96/0.90 → 0.84/0.96/0.90. Метрики змінилися мало, бо обидва нові питання з відповіддю легкі: три методи знаходять їх на першому місці. Трохи знизилися лише hit@5 і MRR у BM25 через перефразоване питання. Питання без відповіді система відхилила, відмов правильних стало 4 з 4.

**28.** За першою таблицею (23 питання): векторний пошук дає hit@1 0.61, hit@5 0.87, MRR 0.74; BM25 — 0.52, 0.65, 0.59; гібридний — 0.61, 0.91, 0.76; з переранжуванням — 0.83, 0.96, 0.90. BM25 найслабший, бо слова в питаннях рідко збігаються з формами слів у документі. Гібридний пошук підняв hit@5 лише з 0.87 до 0.91 (одне питання з 23), а hit@1 не змінив. Переранжування підняло hit@1 з 0.61 до 0.83 (на 5 питань із 23), а MRR — з 0.76 до 0.90. Найбільше воно впливає на hit@1, тому що правильний фрагмент зазвичай уже є серед 20 кандидатів, і проблема в порядку: перший пошук ставить його на 2–5 місце. Cross-encoder бачить запит і фрагмент разом, точніше відрізняє фрагмент із відповіддю від схожого за темою і піднімає його на перше місце. На hit@5 переранжування впливає менше, бо цей показник обмежений кандидатами, яких дав швидкий пошук.

**29.** Генератор і суддя — одна й та сама модель, тому в них спільні слабкі місця: якщо модель неправильно зрозуміла питання або документ, суддя може не помітити цієї помилки. Суддя також не бачить самих документів, а порівнює відповідь лише з еталоном, тому не може перевірити, чи правдиві додаткові подробиці у відповіді. До того ж модель, що оцінює власну відповідь, схильна її схвалювати. Я перевірив п'ять вердиктів вручну. (1) Стипендія ректора: суддя поставив «ні», а я вважаю відповідь правильною, бо вона повторює еталон і додає, що стипендію отримують переможці олімпіад, а це справді є в положенні (я знайшов це речення в тексті). (2) Шифр спеціальності: відповідь «121», еталон F2, суддя поставив «ні», і я згоден, бо відповідь узято зі старої редакції програми. (3) Частка вибіркових дисциплін: «так», я згоден, відповідь збігається з еталоном дослівно. (4) Допомога в академічній відпустці: «так», я згоден, відповідь має ті самі 50 відсотків мінімальної академічної стипендії. (5) Оцінка B: «так», я згоден, 82 – 89 балів. Збіглося 4 вердикти з 5. Розбіжність — випадок 1, де суддя виявився занадто суворим до відповіді, яка містить більше, ніж еталон. Через це реальна частка правильних відповідей вища за 92%: ближче до 96% (24 з 25). Щоб зменшити проблему, суддею можна взяти іншу, сильнішу модель або давати судді ще й знайдені фрагменти.

**30.** У фінальному запуску неправильними визнано дві відповіді. (1) «Для кого встановлюється стипендія ректора?». Ретрив знайшов правильний документ 344 на 1 і 3 місцях, відповідь правильна і процитована правильно ([1] — це документ 344). Тип: помилка судді. Відповідь довша за еталон, але нічому не суперечить, тому реальної помилки системи тут немає. Еталон теж можна було б сформулювати повніше. (2) «Який шифр спеціальності ... в чинній освітній програмі?». Знайдено документи 3329, 3329, 3320, 3329, 3329, тобто правильний документ 3320 серед них був, і це не промах ретриву. Модель відповіла «121» і послалася на фрагмент [2], а це стара редакція програми 3329. Суддя поставив «ні» цілком справедливо. Тип: помилка генерації через суперечливі джерела: стара редакція містить шифр 121, і навіть чинна згадує 121 в абзаці про акредитацію, а шифр F2 серед знайдених збігів стоїть лише в одному місці. Еталон при цьому однозначний, бо чинна редакція — 3320. Виправити це можна, виключивши застарілі документи з пошуку (як у демо 2) або додавши до промпту вказівку віддавати перевагу новішій редакції. Промахів ретриву й неоднозначних еталонів серед помилок немає.